In [1]:
!pip install -U transformers
!pip install datasets accelerate evaluate
!pip install segmentation-models-pytorch
!pip install segmentation-models-pytorch --upgrade
!pip install albumentations --upgrade
!pip install huggingface_hub
!pip install --upgrade huggingface_hub
!pip install git+https://github.com/facebookresearch/detectron2.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 57.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.7/536.7 kB 31.8 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.1
    Uninstalling transformers-4.57.1:
      Successfully uninstalled transformers-4.57.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.1.1 requires transformers<5.0.0,>=4.41.0, but you have transformers 5.0.0 which is incompatible.
gradio 5.49.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.4 MB/s eta

In [2]:
!pip install pillow

In [3]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# Login via CLI
!hf auth login --token $hf_token

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: read).
The token `kaggle` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `kaggle`


In [4]:
import os
import random
import numpy as np
import torch

# =========================================================
# CONFIG (single source of truth)
# =========================================================
class Config:
    # ---------- Dataset ----------
    IMG_DIR   = r"/kaggle/input/plantseg-12k/PlantSeg05/train/images"
    MASK_DIR  = r"/kaggle/input/plantseg-12k/PlantSeg05/train/annotations"

    # Nếu bạn có folder val riêng thì điền, nếu không sẽ split từ train
    VAL_IMG_DIR  = r"/kaggle/input/plantseg-12k/PlantSeg05/val/images"
    VAL_MASK_DIR = "/kaggle/input/plantseg-12k/PlantSeg05/val/annotations"

    NUM_CLASSES = 31
    IMG_SIZE = 512
    IGNORE_INDEX = 255  

    # ---------- Model ----------
    MODEL_NAME = "facebook/dinov3-convnext-small-pretrain-lvd1689m"
    FEATURE_IDXS = [1, 2, 3, 4]                
    BACKBONE_CHANNELS = [96, 192, 384, 768]     

    # ---------- Training ----------
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    SEED = 42

    BATCH_SIZE = 8 
    NUM_EPOCHS = 30

    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 0.01 

    NUM_WORKERS = 2  # Kaggle thường ổn 2-4
    PIN_MEMORY = True if torch.cuda.is_available() else False 

    AMP = True        
    GRAD_CLIP = 1.0   

    # ---------- Normalize ----------
    # Ti số normalize sẽ giống như ImageNet.
    MEAN = (0.485, 0.456, 0.406)
    STD  = (0.229, 0.224, 0.225)

    # ---------- Loss ----------
    # Nên tính từ dataset; tạm thời bạn để như dưới cũng OK.
    CLASS_WEIGHTS = [0.1] + [1.0] * (NUM_CLASSES - 1)

    # ---------- Logging / Saving ----------
    EXP_NAME = "dinov3_convnext_upernet_plantseg" # dòng này có nghĩa là gì? Bạn hãy giải thích.
    SAVE_DIR = "./checkpoints" # dòng này có nghĩa là gì? Bạn hãy giải thích.
    SAVE_BEST = True # dòng này có nghĩa là gì? Bạn hãy giải thích.
    METRIC_FOR_BEST = "mIoU"  # hoặc "mAcc"

    # ---------- Misc ----------
    EPS = 1e-7 # dòng này có nghĩa là gì? Bạn hãy giải thích.


def seed_everything(seed: int = 42): # hàm này có nghĩa là gì? Bạn hãy giải thích.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # giúp reproducible hơn (nhưng có thể giảm speed chút)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(Config.SEED)
os.makedirs(Config.SAVE_DIR, exist_ok=True)
print(f"Config ready. Device: {Config.DEVICE}")


Config ready. Device: cuda


In [5]:
import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ---------- Resize & Pad (mean color for image, ignore_index for mask) ----------
def smart_resize_pad(image, mask, target_size, mask_pad_value=0):
    h, w = image.shape[:2]
    scale = target_size / max(h, w)
    new_h, new_w = int(round(h * scale)), int(round(w * scale))

    # Resize
    resized_image = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    resized_mask  = cv2.resize(mask,  (new_w, new_h), interpolation=cv2.INTER_NEAREST)

    # Mean color (RGB) for padding image
    mean_color = tuple(np.mean(resized_image, axis=(0, 1)).tolist())  # (R, G, B)

    # Padding sizes
    delta_h = target_size - new_h
    delta_w = target_size - new_w
    top = delta_h // 2
    bottom = delta_h - top
    left = delta_w // 2
    right = delta_w - left

    # Pad image with mean color
    padded_image = cv2.copyMakeBorder(
        resized_image, top, bottom, left, right,
        borderType=cv2.BORDER_CONSTANT, value=mean_color
    )

    # Pad mask
    padded_mask = cv2.copyMakeBorder(
        resized_mask, top, bottom, left, right,
        borderType=cv2.BORDER_CONSTANT, value=int(mask_pad_value)
    )

    return padded_image, padded_mask


def get_transforms(phase):
    # Bạn không augment, chỉ normalize + to tensor
    return A.Compose([
        A.Normalize(
            mean=Config.MEAN,
            std=Config.STD,
            max_pixel_value=255.0
        ),
        ToTensorV2()
    ])


class PlantDiseaseDataset(Dataset):
    def __init__(self, images_dir, masks_dir, phase="train", img_size=512, mask_ext=".png"):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.phase = phase
        self.img_size = img_size
        self.mask_ext = mask_ext
        self.transform = get_transforms(phase)

        self.image_ids = sorted([
            f for f in os.listdir(images_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])

        print(f"[{phase.upper()}] Found {len(self.image_ids)} images in {images_dir}")

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_name = self.image_ids[idx]
        img_path = os.path.join(self.images_dir, img_name)

        stem, _ = os.path.splitext(img_name)
        mask_name = stem + self.mask_ext
        mask_path = os.path.join(self.masks_dir, mask_name)

        # Read image (BGR) -> RGB
        image = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if image is None:
            raise FileNotFoundError(f"Cannot read image: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Read mask (grayscale)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise FileNotFoundError(f"Cannot read mask: {mask_path}")

        # Resize + pad to fixed square
        image, mask = smart_resize_pad(
            image=image,
            mask=mask,
            target_size=self.img_size,
            mask_pad_value=Config.IGNORE_INDEX   # ***
        )

        augmented = self.transform(image=image, mask=mask)
        image_tensor = augmented["image"]                 # float32, (3,H,W)
        mask_tensor  = augmented["mask"].long()           # int64, (H,W)

        return image_tensor, mask_tensor


# ---------- Dataloaders ----------
train_ds = PlantDiseaseDataset(
    images_dir=Config.IMG_DIR,
    masks_dir=Config.MASK_DIR,
    phase="train",
    img_size=Config.IMG_SIZE
)

val_ds = PlantDiseaseDataset(
    images_dir=Config.VAL_IMG_DIR,
    masks_dir=Config.VAL_MASK_DIR,
    phase="val",
    img_size=Config.IMG_SIZE
)

train_loader = DataLoader(
    train_ds,
    batch_size=Config.BATCH_SIZE,
    shuffle=True,
    num_workers=Config.NUM_WORKERS,
    pin_memory=Config.PIN_MEMORY,
    drop_last=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=Config.BATCH_SIZE,
    shuffle=False,
    num_workers=Config.NUM_WORKERS,
    pin_memory=Config.PIN_MEMORY,
    drop_last=False
)

print("DataLoaders ready!")


[TRAIN] Found 12000 images in /kaggle/input/plantseg-12k/PlantSeg05/train/images
[VAL] Found 963 images in /kaggle/input/plantseg-12k/PlantSeg05/val/images
DataLoaders ready!


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def norm_layer(num_channels: int, num_groups: int = 32):
    # GroupNorm ổn định cho batch nhỏ
    g = min(num_groups, num_channels)
    # đảm bảo chia hết, nếu không thì giảm group
    while num_channels % g != 0:
        g -= 1
    return nn.GroupNorm(g, num_channels)

class PPM(nn.Module):
    def __init__(self, in_dim, reduction_dim, bins=(1, 2, 3, 6)):
        super().__init__()
        self.bins = bins
        self.stages = nn.ModuleList([
            nn.Sequential(
                nn.AdaptiveAvgPool2d(bin_size),
                nn.Conv2d(in_dim, reduction_dim, kernel_size=1, bias=False),
                norm_layer(reduction_dim),
                nn.ReLU(inplace=True)
            )
            for bin_size in bins
        ])

    def forward(self, x):
        h, w = x.shape[-2:]
        out = [x]
        for stage in self.stages:
            y = stage(x)
            y = F.interpolate(y, size=(h, w), mode="bilinear", align_corners=False)
            out.append(y)
        return torch.cat(out, dim=1)

class UPerNetHead(nn.Module):
    def __init__(self, in_channels, num_classes, fpn_out=256, ppm_bins=(1,2,3,6)):
        super().__init__()
        assert len(in_channels) == 4, "Expected 4 feature levels: [C1,C2,C3,C4]"
        c1, c2, c3, c4 = in_channels

        # PPM on C4
        reduction_dim = c4 // 4
        self.ppm = PPM(c4, reduction_dim, bins=ppm_bins)
        ppm_out_channels = c4 + len(ppm_bins) * reduction_dim

        # Lateral 1x1 convs to fpn_out
        self.lateral_c4 = nn.Sequential(
            nn.Conv2d(ppm_out_channels, fpn_out, kernel_size=1, bias=False),
            norm_layer(fpn_out),
            nn.ReLU(inplace=True),
        )
        self.lateral_c3 = nn.Sequential(
            nn.Conv2d(c3, fpn_out, kernel_size=1, bias=False),
            norm_layer(fpn_out),
            nn.ReLU(inplace=True),
        )
        self.lateral_c2 = nn.Sequential(
            nn.Conv2d(c2, fpn_out, kernel_size=1, bias=False),
            norm_layer(fpn_out),
            nn.ReLU(inplace=True),
        )
        self.lateral_c1 = nn.Sequential(
            nn.Conv2d(c1, fpn_out, kernel_size=1, bias=False),
            norm_layer(fpn_out),
            nn.ReLU(inplace=True),
        )

        # Fuse (concat all levels at C1 resolution)
        self.fuse = nn.Sequential(
            nn.Conv2d(fpn_out * 4, fpn_out, kernel_size=3, padding=1, bias=False),
            norm_layer(fpn_out),
            nn.ReLU(inplace=True),
            nn.Conv2d(fpn_out, num_classes, kernel_size=1)
        )

    def forward(self, features):
        c1, c2, c3, c4 = features

        # PPM + lateral
        p4 = self.lateral_c4(self.ppm(c4))  # (B, fpn_out, 16,16)

        # Top-down
        p3 = self.lateral_c3(c3) + F.interpolate(p4, size=c3.shape[-2:], mode="bilinear", align_corners=False)
        p2 = self.lateral_c2(c2) + F.interpolate(p3, size=c2.shape[-2:], mode="bilinear", align_corners=False)
        p1 = self.lateral_c1(c1) + F.interpolate(p2, size=c1.shape[-2:], mode="bilinear", align_corners=False)

        # Bring all to p1 size then concat
        p2_up = F.interpolate(p2, size=p1.shape[-2:], mode="bilinear", align_corners=False)
        p3_up = F.interpolate(p3, size=p1.shape[-2:], mode="bilinear", align_corners=False)
        p4_up = F.interpolate(p4, size=p1.shape[-2:], mode="bilinear", align_corners=False)

        x = torch.cat([p1, p2_up, p3_up, p4_up], dim=1)
        logits = self.fuse(x)  # (B, num_classes, 128,128) nếu input 512

        return logits


In [7]:
import torch
import torch.nn as nn
from transformers import AutoModel

class PlantSegmentor(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

        # Backbone
        print(f"--- Loading Backbone: {cfg.MODEL_NAME} ---")
        self.backbone = AutoModel.from_pretrained(
            cfg.MODEL_NAME,
            trust_remote_code=True,
            output_hidden_states=True
        )

        # Freeze backbone
        for p in self.backbone.parameters():
            p.requires_grad = False
        self.backbone.eval()  # important when frozen

        # Head (custom UPerNetHead you already have)
        self.head = UPerNetHead(
            in_channels=cfg.BACKBONE_CHANNELS,
            num_classes=cfg.NUM_CLASSES
        )

    def forward(self, x):
        # x: (B,3,H,W)
        # Extract multi-scale features without grad to save VRAM
        with torch.no_grad():
            outputs = self.backbone(pixel_values=x)
            hs = outputs.hidden_states

        feats = [hs[i] for i in self.cfg.FEATURE_IDXS]  # [C1,C2,C3,C4]
        logits = self.head(feats)  # (B, num_classes, h, w) (usually at 1/4 or 1/8, tùy head)

        logits = nn.functional.interpolate(
            logits,
            size=x.shape[-2:],
            mode="bilinear",
            align_corners=False
        )
        return logits


In [8]:
import torch
import torch.nn as nn

def build_loss(cfg):
    # weight: tensor shape (C,)
    weights = torch.tensor(cfg.CLASS_WEIGHTS, dtype=torch.float32, device=cfg.DEVICE)
    loss_fn = nn.CrossEntropyLoss(
        weight=weights,
        ignore_index=cfg.IGNORE_INDEX
    )
    return loss_fn

def build_optimizer(cfg, model):
    # Backbone đã freeze, optimizer chỉ update head
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        params,
        lr=cfg.LEARNING_RATE,
        weight_decay=cfg.WEIGHT_DECAY
    )
    return optimizer

# Optional: scheduler (cosine)
def build_scheduler(cfg, optimizer, num_training_steps):
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_training_steps
    )
    return scheduler


In [9]:
@torch.no_grad()
def confusion_matrix_from_logits(logits, target, num_classes, ignore_index=255):
    # logits: (B,C,H,W), target: (B,H,W)
    pred = torch.argmax(logits, dim=1).view(-1)
    target = target.view(-1)

    if ignore_index is not None:
        valid = target != ignore_index
        pred = pred[valid]
        target = target[valid]

    valid = (target >= 0) & (target < num_classes)
    pred = pred[valid]
    target = target[valid]

    idx = target * num_classes + pred
    cm = torch.bincount(idx, minlength=num_classes * num_classes).reshape(num_classes, num_classes)
    return cm

@torch.no_grad()
def compute_miou_macc_from_confmat(confmat, eps=1e-7, ignore_empty=True):
    confmat = confmat.to(torch.float32)
    tp = torch.diag(confmat)
    gt_sum = confmat.sum(dim=1)   # TP + FN
    pred_sum = confmat.sum(dim=0) # TP + FP

    fp = pred_sum - tp
    fn = gt_sum - tp

    iou = tp / (tp + fp + fn + eps)
    acc = tp / (tp + fn + eps)  # mAcc theo công thức bạn đưa (mean per-class recall)

    if ignore_empty:
        valid = gt_sum > 0
        miou = iou[valid].mean().item() if valid.any() else 0.0
        macc = acc[valid].mean().item() if valid.any() else 0.0
    else:
        miou = iou.mean().item()
        macc = acc.mean().item()

    return miou, macc, iou, acc

In [10]:
import os
import torch
from tqdm.auto import tqdm

# ---------- AMP helpers (version-safe) ----------
def get_autocast(enabled: bool):
    # torch.autocast("cuda") tương thích tốt trên nhiều version
    if enabled and torch.cuda.is_available():
        return torch.autocast("cuda", enabled=True)
    return torch.autocast("cpu", enabled=False)

def get_scaler(enabled: bool):
    if enabled and torch.cuda.is_available():
        try:
            from torch.amp import GradScaler
            return GradScaler(enabled=True)
        except Exception:
            from torch.cuda.amp import GradScaler
            return GradScaler(enabled=True)
    return None


def train_one_epoch(cfg, model, loader, loss_fn, optimizer, scaler, epoch_idx):
    model.train()
    use_amp = cfg.AMP and torch.cuda.is_available()

    running_loss = 0.0
    pbar = tqdm(loader, desc=f"Train Epoch {epoch_idx}", leave=False)

    for images, masks in pbar:
        images = images.to(cfg.DEVICE, non_blocking=True)
        masks  = masks.to(cfg.DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with get_autocast(use_amp):
            logits = model(images)
            loss = loss_fn(logits, masks)

        if use_amp:
            scaler.scale(loss).backward()
            if cfg.GRAD_CLIP and cfg.GRAD_CLIP > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            if cfg.GRAD_CLIP and cfg.GRAD_CLIP > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            optimizer.step()

        bs = images.size(0)
        running_loss += loss.item() * bs
        avg_loss = running_loss / ((pbar.n + 1) * bs)

        pbar.set_postfix(loss=float(loss.item()), avg_loss=float(avg_loss))

    epoch_loss = running_loss / len(loader.dataset)
    return epoch_loss


@torch.no_grad()
def validate(cfg, model, loader, loss_fn, epoch_idx):
    model.eval()
    use_amp = cfg.AMP and torch.cuda.is_available()

    running_loss = 0.0
    confmat = torch.zeros(cfg.NUM_CLASSES, cfg.NUM_CLASSES, device=cfg.DEVICE, dtype=torch.int64)

    pbar = tqdm(loader, desc=f"Val   Epoch {epoch_idx}", leave=False)

    for images, masks in pbar:
        images = images.to(cfg.DEVICE, non_blocking=True)
        masks  = masks.to(cfg.DEVICE, non_blocking=True)

        with get_autocast(use_amp):
            logits = model(images)
            loss = loss_fn(logits, masks)

        running_loss += loss.item() * images.size(0)
        confmat += confusion_matrix_from_logits(logits, masks, cfg.NUM_CLASSES, cfg.IGNORE_INDEX)

        pbar.set_postfix(val_loss=float(loss.item()))

    val_loss = running_loss / len(loader.dataset)
    miou, macc, _, _ = compute_miou_macc_from_confmat(confmat, eps=cfg.EPS, ignore_empty=True)
    return val_loss, miou, macc


def fit(cfg, model, train_loader, val_loader):
    os.makedirs(cfg.SAVE_DIR, exist_ok=True)

    loss_fn = build_loss(cfg)
    optimizer = build_optimizer(cfg, model)

    use_amp = cfg.AMP and torch.cuda.is_available()
    scaler = get_scaler(use_amp)

    best_metric = -1.0

    for epoch in range(1, cfg.NUM_EPOCHS + 1):
        train_loss = train_one_epoch(cfg, model, train_loader, loss_fn, optimizer, scaler, epoch)
        val_loss, miou, macc = validate(cfg, model, val_loader, loss_fn, epoch)

        # chọn metric để save best
        metric = miou if cfg.METRIC_FOR_BEST.lower() == "miou" else macc

        # ---- Epoch report (bạn muốn) ----
        print(
            f"[Epoch {epoch:02d}/{cfg.NUM_EPOCHS}] "
            f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
            f"mIoU={miou:.4f} | mAcc={macc:.4f}"
        )

        if cfg.SAVE_BEST and metric > best_metric:
            best_metric = metric
            ckpt_path = os.path.join(cfg.SAVE_DIR, f"{cfg.EXP_NAME}_best.pt")
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_metric": best_metric,
                },
                ckpt_path
            )
            print(f"✅ Saved best: {ckpt_path} ({cfg.METRIC_FOR_BEST}={best_metric:.4f})")


In [11]:
model = PlantSegmentor(Config).to(Config.DEVICE)

--- Loading Backbone: facebook/dinov3-convnext-small-pretrain-lvd1689m ---


config.json:   0%|          | 0.00/447 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/198M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/342 [00:00<?, ?it/s]

In [12]:
fit(Config, model, train_loader, val_loader)

Train Epoch 1:   0%|          | 0/1500 [00:00<?, ?it/s]

Val   Epoch 1:   0%|          | 0/121 [00:00<?, ?it/s]

[Epoch 01/30] train_loss=1.0746 | val_loss=0.6721 | mIoU=0.3656 | mAcc=0.7567
✅ Saved best: ./checkpoints/dinov3_convnext_upernet_plantseg_best.pt (mIoU=0.3656)


Train Epoch 2:   0%|          | 0/1500 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7fe8f1f33ba0>
Traceback (most recent call last):
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7fe8f1f33ba0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1610, in _shutdown_workers
    self._pin_memory_thread.join()
  File "/usr/lib/python3.12/threading.py", line 1146, in join
    raise RuntimeError("cannot join current thread")
RuntimeError: cannot join current thread
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1647, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^Exception ignored in: ^^<functi

...

[Epoch 29/30] train_loss=0.1051 | val_loss=0.8290 | mIoU=0.5329 | mAcc=0.7652


Train Epoch 30:   0%|          | 0/1500 [00:00<?, ?it/s]

Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7fe8f1f33ba0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1647, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^Exception ignored in: ^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7fe8f1f33ba0>
Traceback (most recent call last):

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    self._shutdown_workers()    assert self._parent_pid == os.getpid(), 'can only test a child process'

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1647, in _shutdown_workers
    if w.is_alive(): 
               ^ ^^ ^^^^^^^^^^^^^^^^^^^^^

Val   Epoch 30:   0%|          | 0/121 [00:00<?, ?it/s]

[Epoch 30/30] train_loss=0.1105 | val_loss=0.8201 | mIoU=0.5415 | mAcc=0.7647
